# HeatWise AI: demo notebook

HeatWise AI is a multi-agent heatwave preparedness assistant built with the Strands Agents SDK. A coordinator agent delegates to four specialists (weather, community risk, action planner, safety and verification). The agents run on Amazon Bedrock AgentCore Runtime with Amazon Nova 2 Lite.

This notebook calls the deployed job API, runs one assessment, and prints the verified result.

- Code: https://github.com/dineshrajdhanapathyDD/HeatWise-AI
- Dashboard: https://main.dmupjpltx9cou.amplifyapp.com

It is an informational and preparedness tool. It is not medical advice and not an official warning. A run takes about 20-25 seconds. The API is a demo deployment and may be turned off after the judging window.

The only dependency is `requests`. No AWS credentials are needed.

In [ ]:
%pip install -q requests

In [ ]:
import json
import time

import requests

API = "https://pwnenvmo71.execute-api.us-east-1.amazonaws.com"

health = requests.get(f"{API}/api/health", timeout=30)
health.raise_for_status()
print(json.dumps(health.json(), indent=2))

## 1. Start an assessment

The request has only a place, the forecast length and yes/no community indicators. Any other field (a name, a phone number) is rejected.

In [ ]:
request_body = {
    "place": {"name": "Thanjavur", "state": "Tamil Nadu"},
    "forecast_days": 3,
    "profile": {"elderly": True, "outdoor_workers": True, "farmers": True},
}

created = requests.post(f"{API}/api/assessments", json=request_body, timeout=30)
created.raise_for_status()
job = created.json()
print("job_id:", job["job_id"], "| status:", job["status"])

## 2. Wait for the job

The API returns a job id at once and the agents run in the background, because an HTTP API request times out at 30 seconds.

In [ ]:
deadline = time.time() + 180
while True:
    job = requests.get(f"{API}/api/assessments/{job['job_id']}", timeout=30).json()
    print(job["status"])
    if job["status"] in ("succeeded", "failed") or time.time() > deadline:
        break
    time.sleep(3)

print("duration_ms:", job.get("duration_ms"))
print("error:", job.get("error"))
print("report_id:", job.get("report_id"))

## 3. Read the stored report

The stored record holds the full report. The first cell lists its fields, so you can see what the agents produced.

In [ ]:
assert job.get("report_id"), "The job did not produce a report. See the error above."

record = requests.get(f"{API}/api/reports/{job['report_id']}", timeout=30).json()
print("record fields:", sorted(record.keys()))
report = record.get("report", record)
print("report fields:", sorted(report.keys()))

In [ ]:
# Status, risk and verification. The risk category comes from documented, deterministic rules, not from the model.
for key in ("status", "risk", "verification"):
    if key in report:
        print(f"--- {key}")
        print(json.dumps(report[key], indent=2, default=str)[:1500])

In [ ]:
# The checklist a volunteer can print or share. Only verified items appear in it.
print(report.get("checklist_markdown", "No checklist in this report."))

## 4. Which agents ran, in what order

The trace shows each agent step. Hooks block unapproved tool calls and a steering handler keeps the order weather, risk, plan, safety.

In [ ]:
for step in report.get("trace", []):
    print(json.dumps(step, default=str)[:200])

## 5. Synthetic evaluation results

These are results on synthetic test scenarios, each labelled with the mode it was measured in. They are not observations of real communities, and no health outcome is measured or claimed.

In [ ]:
ev = requests.get(f"{API}/api/evaluation", timeout=30)
print(ev.status_code)
print(json.dumps(ev.json(), indent=2)[:3000])